In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark import StorageLevel
import time

spark = SparkSession.builder \
    .appName("FraudDetection_SQL_Optimize") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

schema = StructType([
    StructField("Transaction ID",       StringType(),  True),
    StructField("Customer ID",          StringType(),  True),
    StructField("Transaction Amount",   DoubleType(),  True),
    StructField("Transaction Date",     StringType(),  True),
    StructField("Payment Method",       StringType(),  True),
    StructField("Product Category",     StringType(),  True),
    StructField("Quantity",             IntegerType(), True),
    StructField("Customer Age",         IntegerType(), True),
    StructField("Customer Location",    StringType(),  True),
    StructField("Device Used",          StringType(),  True),
    StructField("IP Address",           StringType(),  True),
    StructField("Shipping Address",     StringType(),  True),
    StructField("Billing Address",      StringType(),  True),
    StructField("Is Fraudulent",        IntegerType(), True),
    StructField("Account Age Days",     IntegerType(), True),
    StructField("Transaction Hour",     IntegerType(), True),
])

In [2]:
# Đọc DataFrame
df = spark.read.csv(
    "hdfs://localhost:9000/final/fraud_300k_raw_temporal.csv",
    header=True, schema=schema, multiLine=True, escape='"'
)

# Lần 1: count() TRƯỚC persist — Spark đọc lại từ HDFS
t0 = time.time()
df.count()
t1 = time.time()
time_before = round(t1 - t0, 3)
print(f"[TRƯỚC PERSIST] count() từ HDFS: {time_before}s")

# Apply persist + materialize
df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()  # materialize — không đo lần này

# Lần 2: count() SAU persist — Spark đọc từ RAM
t2 = time.time()
df.count()
t3 = time.time()
time_after = round(t3 - t2, 3)
print(f"[SAU PERSIST]  count() từ cache: {time_after}s")
print(f"Cải thiện: {round(time_before/time_after, 1)}x")

df.createOrReplaceTempView("transactions")

[TRƯỚC PERSIST] count() từ HDFS: 5.335s
[SAU PERSIST]  count() từ cache: 0.393s
Cải thiện: 13.6x


In [3]:
print("=" * 60)
print("PHYSICAL PLAN - df từ HDFS với persist:")
print("=" * 60)
df.explain(True)

PHYSICAL PLAN - df từ HDFS với persist:
== Parsed Logical Plan ==
UnresolvedDataSource format: csv, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
Transaction ID: string, Customer ID: string, Transaction Amount: double, Transaction Date: string, Payment Method: string, Product Category: string, Quantity: int, Customer Age: int, Customer Location: string, Device Used: string, IP Address: string, Shipping Address: string, Billing Address: string, Is Fraudulent: int, Account Age Days: int, Transaction Hour: int
Relation [Transaction ID#0,Customer ID#1,Transaction Amount#2,Transaction Date#3,Payment Method#4,Product Category#5,Quantity#6,Customer Age#7,Customer Location#8,Device Used#9,IP Address#10,Shipping Address#11,Billing Address#12,Is Fraudulent#13,Account Age Days#14,Transaction Hour#15] csv

== Optimized Logical Plan ==
InMemoryRelation [Transaction ID#0, Customer ID#1, Transaction Amount#2, Transaction Date#3, Payment Method#4, Product Category#5, Quantity#6, C

In [4]:
result1 = spark.sql("""
    SELECT `Payment Method`,
           COUNT(*) AS total_transactions,
           SUM(CASE WHEN `Is Fraudulent` = 1 THEN 1 ELSE 0 END) AS fraud_count,
           ROUND(AVG(`Transaction Amount`), 2) AS avg_amount,
           ROUND(SUM(CASE WHEN `Is Fraudulent` = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fraud_rate_pct
    FROM transactions
    GROUP BY `Payment Method`
    ORDER BY fraud_rate_pct DESC
""")
result1.show()


+--------------+------------------+-----------+----------+--------------+
|Payment Method|total_transactions|fraud_count|avg_amount|fraud_rate_pct|
+--------------+------------------+-----------+----------+--------------+
|        PayPal|             74935|       3791|    226.76|          5.06|
|    debit card|             75053|       3764|    226.56|          5.02|
| bank transfer|             74912|       3749|    225.21|          5.00|
|   credit card|             75100|       3690|    227.46|          4.91|
+--------------+------------------+-----------+----------+--------------+



In [5]:
result2 = spark.sql("""
    WITH weekly AS (
        SELECT
            COALESCE(
                try_to_date(`Transaction Date`, 'M/d/yyyy H:mm'),
                try_to_date(`Transaction Date`, 'yyyy-MM-dd HH:mm:ss')
            ) AS tx_date,
            `Is Fraudulent`,
            `Transaction Amount`
        FROM transactions
    )
    SELECT
        YEAR(tx_date) AS year_num,
        WEEKOFYEAR(tx_date) AS week_number,
        COUNT(*) AS total_transactions,
        SUM(`Is Fraudulent`) AS fraud_count,
        ROUND(SUM(`Is Fraudulent`) * 100.0 / COUNT(*), 2) AS fraud_rate_pct,
        ROUND(SUM(`Transaction Amount` * `Is Fraudulent`), 2) AS total_fraud_amount
    FROM weekly
    GROUP BY YEAR(tx_date), WEEKOFYEAR(tx_date)
    ORDER BY year_num, week_number
""")
result2.show()

+--------+-----------+------------------+-----------+--------------+------------------+
|year_num|week_number|total_transactions|fraud_count|fraud_rate_pct|total_fraud_amount|
+--------+-----------+------------------+-----------+--------------+------------------+
|    2024|          5|            102548|       5113|          4.99|        2788700.65|
|    2024|          6|            110387|       5557|          5.03|        3085432.43|
|    2024|          7|             87065|       4324|          4.97|        2371066.51|
+--------+-----------+------------------+-----------+--------------+------------------+



In [6]:
result3 = spark.sql("""
    SELECT address_match_status,
           COUNT(*) AS total_transactions,
           SUM(`Is Fraudulent`) AS fraud_count,
           ROUND(SUM(`Is Fraudulent`) * 100.0 / COUNT(*), 2) AS fraud_rate_pct,
           ROUND(AVG(`Transaction Amount`), 2) AS avg_amount
    FROM (
        SELECT *,
               CASE
                   WHEN `Shipping Address` = `Billing Address` THEN 'Địa chỉ khớp'
                   ELSE 'Địa chỉ không khớp'
               END AS address_match_status
        FROM transactions
    ) addr
    GROUP BY address_match_status
    ORDER BY fraud_rate_pct DESC
""")
result3.show()


+--------------------+------------------+-----------+--------------+----------+
|address_match_status|total_transactions|fraud_count|fraud_rate_pct|avg_amount|
+--------------------+------------------+-----------+--------------+----------+
|  Địa chỉ không khớp|             30046|       1512|          5.03|    228.25|
|        Địa chỉ khớp|            269954|      13482|          4.99|     226.3|
+--------------------+------------------+-----------+--------------+----------+



In [7]:
result4 = spark.sql("""
    SELECT `Transaction ID`, `Product Category`, `Transaction Amount`, rank_in_category
    FROM (
        SELECT `Transaction ID`,
               `Product Category`,
               `Transaction Amount`,
               RANK() OVER (PARTITION BY `Product Category` ORDER BY `Transaction Amount` DESC) AS rank_in_category
        FROM transactions
        WHERE `Is Fraudulent` = 1
    ) ranked
    WHERE rank_in_category <= 5
    ORDER BY `Product Category`, rank_in_category
""")
result4.show()


+--------------------+----------------+------------------+----------------+
|      Transaction ID|Product Category|Transaction Amount|rank_in_category|
+--------------------+----------------+------------------+----------------+
|5c0b2b44-e588-44a...|        clothing|            6597.5|               1|
|7a58c5b4-fdd0-461...|        clothing| 6421.799999999999|               2|
|27b2ab78-2239-460...|        clothing| 5491.799999999999|               3|
|d6cf7f93-9a59-484...|        clothing| 5465.549999999999|               4|
|ee06e1ba-2200-456...|        clothing|            5191.4|               5|
|e60e9860-8a94-45d...|     electronics|           6847.55|               1|
|1f223be2-d8af-41d...|     electronics|           6611.35|               2|
|2adb485d-4230-4c2...|     electronics|           5736.05|               3|
|1bf8ddaf-e75f-4e5...|     electronics|            5554.8|               4|
|92730fb2-a0af-4a9...|     electronics|            5527.7|               5|
|c0dde840-04

In [8]:
result5 = spark.sql("""
    SELECT age_group,
           COUNT(*) AS total_transactions,
           SUM(`Is Fraudulent`) AS fraud_count,
           ROUND(SUM(`Is Fraudulent`) * 100.0 / COUNT(*), 2) AS fraud_rate_pct,
           ROUND(AVG(`Transaction Amount`), 2) AS avg_amount,
           RANK() OVER (ORDER BY SUM(`Is Fraudulent`) * 100.0 / COUNT(*) DESC) AS risk_rank
    FROM (
        SELECT *,
               CASE
                   WHEN `Customer Age` < 25 THEN 'Trẻ (< 25)'
                   WHEN `Customer Age` < 35 THEN 'Thanh niên (25-34)'
                   WHEN `Customer Age` < 50 THEN 'Trung niên (35-49)'
                   ELSE 'Lớn tuổi (50+)'
               END AS age_group
        FROM transactions
    ) aged
    GROUP BY age_group
    ORDER BY risk_rank
""")
result5.show()


+------------------+------------------+-----------+--------------+----------+---------+
|         age_group|total_transactions|fraud_count|fraud_rate_pct|avg_amount|risk_rank|
+------------------+------------------+-----------+--------------+----------+---------+
|        Trẻ (< 25)|             47530|       2435|          5.12|    225.29|        1|
|Trung niên (35-49)|            130020|       6477|          4.98|    226.22|        2|
|Thanh niên (25-34)|            102447|       5100|          4.98|    227.29|        3|
|    Lớn tuổi (50+)|             20003|        982|          4.91|    227.08|        4|
+------------------+------------------+-----------+--------------+----------+---------+



In [9]:
result6 = spark.sql("""
WITH category_region_stats AS (
    SELECT
        `Customer Location` AS region_proxy,
        `Product Category`,
        COUNT(*) AS total_transactions,
        SUM(`Is Fraudulent`) AS fraud_transactions,
        ROUND(SUM(`Is Fraudulent`) / COUNT(*), 4) AS fraud_rate
    FROM transactions
    GROUP BY `Customer Location`, `Product Category`
    HAVING COUNT(*) >= 5
),


ranked_categories AS (
    SELECT
        region_proxy,
        `Product Category`,
        total_transactions,
        fraud_transactions,
        fraud_rate,
        RANK() OVER (
            PARTITION BY region_proxy
            ORDER BY fraud_rate DESC, total_transactions DESC
        ) AS category_rank
    FROM category_region_stats
)


SELECT
    region_proxy,
    `Product Category`,
    total_transactions,
    fraud_transactions,
    fraud_rate,
    category_rank
FROM ranked_categories
WHERE category_rank <= 3
ORDER BY region_proxy, category_rank
""")
result6.show(50, truncate=False)


+------------+----------------+------------------+------------------+----------+-------------+
|region_proxy|Product Category|total_transactions|fraud_transactions|fraud_rate|category_rank|
+------------+----------------+------------------+------------------+----------+-------------+
|Aaronberg   |clothing        |5                 |0                 |0.0       |1            |
|Aaronburgh  |clothing        |6                 |0                 |0.0       |1            |
|Aaronburgh  |health & beauty |5                 |0                 |0.0       |2            |
|Aaronchester|toys & games    |5                 |0                 |0.0       |1            |
|Aaronhaven  |health & beauty |7                 |0                 |0.0       |1            |
|Aaronland   |toys & games    |5                 |0                 |0.0       |1            |
|Aaronmouth  |home & garden   |10                |0                 |0.0       |1            |
|Aaronmouth  |electronics     |9                 |

In [10]:
result7 = spark.sql("""
    WITH daily_fraud AS (
        SELECT
            COALESCE(
                try_to_date(`Transaction Date`, 'M/d/yyyy H:mm'),
                try_to_date(`Transaction Date`, 'yyyy-MM-dd HH:mm:ss')
            ) AS tx_date,
            `Is Fraudulent`,
            `Transaction Amount`
        FROM transactions
    ),
    daily_stats AS (
        SELECT
            tx_date,
            COUNT(*) AS total_transactions,
            SUM(`Is Fraudulent`) AS fraud_transactions,
            ROUND(SUM(`Is Fraudulent`) / COUNT(*), 4) AS daily_fraud_rate
        FROM daily_fraud
        GROUP BY tx_date
    ),
    daily_moving_avg AS (
        SELECT
            tx_date,
            total_transactions,
            fraud_transactions,
            daily_fraud_rate,
            ROUND(
                AVG(daily_fraud_rate) OVER (
                    ORDER BY tx_date
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ), 4
            ) AS moving_avg_7d_fraud_rate
        FROM daily_stats
    )
    SELECT *
    FROM daily_moving_avg
    ORDER BY tx_date
""")
result7.show()

+----------+------------------+------------------+----------------+------------------------+
|   tx_date|total_transactions|fraud_transactions|daily_fraud_rate|moving_avg_7d_fraud_rate|
+----------+------------------+------------------+----------------+------------------------+
|2024-01-29|              7806|               386|          0.0494|                  0.0494|
|2024-01-30|             15628|               794|          0.0508|                  0.0501|
|2024-01-31|             15593|               741|          0.0475|                  0.0492|
|2024-02-01|             15678|               791|          0.0505|                  0.0496|
|2024-02-02|             15751|               803|           0.051|                  0.0498|
|2024-02-03|             16027|               805|          0.0502|                  0.0499|
|2024-02-04|             16065|               793|          0.0494|                  0.0498|
|2024-02-05|             15649|               776|          0.0496|   

In [11]:
result8 = spark.sql("""
WITH payment_stats AS (
    SELECT
        `Payment Method`,
        COUNT(*) AS total_transactions,
        SUM(`Is Fraudulent`) AS fraud_transactions,
        SUM(`Is Fraudulent`) / COUNT(*) AS payment_fraud_rate
    FROM transactions
    GROUP BY `Payment Method`
)


SELECT
    `Payment Method`,
    total_transactions,
    fraud_transactions,
    ROUND(payment_fraud_rate, 4) AS payment_fraud_rate,
    ROUND((SELECT AVG(`Is Fraudulent`) FROM transactions), 4) AS global_fraud_rate,
    ROUND(payment_fraud_rate - (SELECT AVG(`Is Fraudulent`) FROM transactions), 4) AS fraud_rate_gap
FROM payment_stats
WHERE payment_fraud_rate > (
    SELECT AVG(`Is Fraudulent`)
    FROM transactions
)
ORDER BY fraud_rate_gap DESC
""")
result8.show(truncate=False)


+--------------+------------------+------------------+------------------+-----------------+--------------+
|Payment Method|total_transactions|fraud_transactions|payment_fraud_rate|global_fraud_rate|fraud_rate_gap|
+--------------+------------------+------------------+------------------+-----------------+--------------+
|PayPal        |74935             |3791              |0.0506            |0.05             |6.0E-4        |
|debit card    |75053             |3764              |0.0502            |0.05             |2.0E-4        |
|bank transfer |74912             |3749              |0.05              |0.05             |1.0E-4        |
+--------------+------------------+------------------+------------------+-----------------+--------------+



In [12]:
result9 = spark.sql("""
WITH cohort_amount AS (
    SELECT
        `Transaction ID`,
        `Customer ID`,
        `Transaction Date`,
        `Payment Method`,
        `Product Category`,
        `Device Used`,
        `Transaction Amount`,
        AVG(`Transaction Amount`) OVER (
            PARTITION BY `Payment Method`, `Product Category`, `Device Used`
        ) AS avg_cohort_amount,
        `Is Fraudulent`
    FROM transactions
)
SELECT
    `Transaction ID`,
    `Customer ID`,
    `Transaction Date`,
    `Payment Method`,
    `Product Category`,
    `Device Used`,
    ROUND(`Transaction Amount`, 2) AS transaction_amount,
    ROUND(avg_cohort_amount, 2) AS avg_cohort_amount,
    ROUND(`Transaction Amount` / avg_cohort_amount, 2) AS amount_ratio,
    `Is Fraudulent`
FROM cohort_amount
WHERE `Transaction Amount` > 2 * avg_cohort_amount
ORDER BY amount_ratio DESC
""")
result9.show(50, truncate=False)


+------------------------------------+------------------------------------+-------------------+--------------+----------------+-----------+------------------+-----------------+------------+-------------+
|Transaction ID                      |Customer ID                         |Transaction Date   |Payment Method|Product Category|Device Used|transaction_amount|avg_cohort_amount|amount_ratio|Is Fraudulent|
+------------------------------------+------------------------------------+-------------------+--------------+----------------+-----------+------------------+-----------------+------------+-------------+
|d4b6f664-c05b-428b-9957-60a3de4ce7aa|c82ed907-419e-43d7-ab8f-9ccbdbd1225c|2024-02-07 10:39:33|bank transfer |toys & games    |mobile     |7746.65           |225.57           |34.34       |1            |
|c0dde840-04bc-4504-8f55-df9c2afabdc2|423a4f55-5f9c-4f9e-8f01-f65b5341ba63|2024-02-12 11:36:59|bank transfer |health & beauty |mobile     |7292.2            |223.31           |32.65   

In [13]:
result10 = spark.sql("""
WITH risk_profile AS (
    SELECT
        *,
        CASE
            WHEN `Account Age Days` <= 87 THEN 'new'
            WHEN `Account Age Days` <= 180 THEN 'young'
            WHEN `Account Age Days` <= 273 THEN 'mature'
            ELSE 'veteran'
        END AS account_age_bucket,


        CASE
            WHEN `Transaction Hour` IN (0, 1, 2, 3, 4) THEN 1
            ELSE 0
        END AS is_night,


        CASE
            WHEN `Shipping Address` != `Billing Address` THEN 1
            ELSE 0
        END AS address_mismatch
    FROM transactions
)
SELECT
    account_age_bucket,
    is_night,
    address_mismatch,
    COUNT(*) AS total_transactions,
    SUM(`Is Fraudulent`) AS fraud_transactions,
    ROUND(SUM(`Is Fraudulent`) / COUNT(*), 4) AS fraud_rate,
    ROUND(AVG(`Transaction Amount`), 2) AS avg_transaction_amount,
    ROUND(SUM(`Transaction Amount`), 2) AS total_transaction_amount
FROM risk_profile
GROUP BY
    account_age_bucket,
    is_night,
    address_mismatch
HAVING COUNT(*) >= 100
ORDER BY
    fraud_rate DESC,
    total_transactions DESC
""")
result10.show(50, truncate=False)


+------------------+--------+----------------+------------------+------------------+----------+----------------------+------------------------+
|account_age_bucket|is_night|address_mismatch|total_transactions|fraud_transactions|fraud_rate|avg_transaction_amount|total_transaction_amount|
+------------------+--------+----------------+------------------+------------------+----------+----------------------+------------------------+
|new               |1       |1               |1803              |395               |0.2191    |278.15                |501506.6                |
|new               |1       |0               |15906             |3319              |0.2087    |277.69                |4416985.02              |
|new               |0       |1               |5772              |436               |0.0755    |239.15                |1380388.67              |
|new               |0       |0               |52102             |3877              |0.0744    |234.43                |1.221415132E7     